<div style="display:flex;align-items:center;justify-content:space-between;border-bottom:2px solid #c8962d;padding-bottom:12px;margin-bottom:20px">
  <div>
    <strong>Universidad Externado de Colombia</strong><br>
    <span>Programa de Ciencia de Datos · Machine Learning II</span><br>
    <span>Docente: Wilmer Pineda-Ríos</span>
  </div>
  <img src="../../assets/brand/logo-externado.png" alt="Universidad Externado de Colombia" width="190" onerror="this.style.display='none'">
</div>

# Semana 1 — Árboles de decisión para clasificación

**Objetivo.** Construir, evaluar e interpretar un árbol para anticipar churn, sin usar el conjunto de prueba para escoger su complejidad.

## 1. Decisión y datos

El equipo de retención debe priorizar clientes con riesgo de cancelar. Un falso negativo deja pasar un cliente que efectivamente se irá; un falso positivo consume capacidad de contacto. Usaremos **recall de churn** como métrica principal y precision/F1 como contrapeso.

Fuente: *Iranian Churn Dataset*, UCI Machine Learning Repository, DOI `10.24432/C5JW3Z`, licencia CC BY 4.0. Las variables resumen los primeros nueve meses y la etiqueta registra el estado al final del periodo, dejando una ventana de planeación.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (ConfusionMatrixDisplay, classification_report, f1_score,
                             make_scorer, precision_score, recall_score)
from sklearn.model_selection import StratifiedKFold, cross_validate, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier, plot_tree

RANDOM_STATE = 42
sns.set_theme(style="whitegrid")

In [ ]:
candidates = [
    Path("../../datasets/public/Customer_Churn_Iran.csv"),
    Path("datasets/public/Customer_Churn_Iran.csv"),
]
data_path = next((path for path in candidates if path.exists()), None)
if data_path is None:
    raise FileNotFoundError("Ejecute el notebook desde su carpeta o desde la raíz del curso.")

df = pd.read_csv(data_path)
df.columns = (df.columns.str.strip().str.lower().str.replace(r"\s+", "_", regex=True))
df.shape, df.head()

### Diccionario mínimo

`churn` es la clase objetivo (1: cancela; 0: permanece). Entre los predictores están fallas de llamadas, quejas, antigüedad, uso, plan, estado y valor del cliente. No hay valores faltantes en la fuente.

In [ ]:
quality = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "missing": df.isna().sum(),
    "unique": df.nunique(),
})
display(quality)
display(df["churn"].value_counts().rename("n").to_frame().assign(share=lambda x: x["n"] / len(df)))

## 2. Partición antes de modelar

Reservamos 20 % para una evaluación final. La búsqueda de complejidad ocurrirá únicamente dentro del entrenamiento mediante validación cruzada estratificada.

In [ ]:
X = df.drop(columns="churn")
y = df["churn"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=RANDOM_STATE
)
pd.DataFrame({"train": y_train.value_counts(normalize=True), "test": y_test.value_counts(normalize=True)})

## 3. Puente con ML1: referencias

Comparamos contra la clase mayoritaria y una regresión logística. La regresión logística sí recibe escalamiento dentro de un `Pipeline`; el árbol trabajará después con las variables originales.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
scoring = {
    "precision": make_scorer(precision_score, zero_division=0),
    "recall": make_scorer(recall_score, zero_division=0),
    "f1": make_scorer(f1_score, zero_division=0),
}
references = {
    "Mayoritaria": DummyClassifier(strategy="most_frequent"),
    "Logística": Pipeline([
        ("scale", StandardScaler()),
        ("model", LogisticRegression(max_iter=2000, random_state=RANDOM_STATE)),
    ]),
}
reference_rows = []
for name, model in references.items():
    scores = cross_validate(model, X_train, y_train, cv=cv, scoring=scoring)
    reference_rows.append({"modelo": name, **{m: scores[f"test_{m}"].mean() for m in scoring}})
pd.DataFrame(reference_rows).set_index("modelo").round(3)

## 4. Gini a mano

Antes de ajustar el modelo, calculemos la impureza de un nodo. Para dos clases, $G=1-p_0^2-p_1^2$.

In [ ]:
def gini(counts):
    proportions = np.asarray(counts) / np.sum(counts)
    return 1 - np.sum(proportions**2)

examples = pd.DataFrame({
    "nodo": ["50/50", "90/10", "100/0"],
    "gini": [gini([50, 50]), gini([90, 10]), gini([100, 0])],
})
examples

**Reto 1.** Una división produce un nodo izquierdo con `[80, 20]` y uno derecho con `[10, 40]`. Calcule el Gini ponderado y compárelo con el nodo padre `[90, 60]`. ¿La división reduce impureza?

In [ ]:
# Complete el cálculo
gini_parent = gini([90, 60])
gini_split = (100 / 150) * gini([80, 20]) + (50 / 150) * gini([10, 40])
gini_parent, gini_split, gini_parent - gini_split

## 5. Un árbol interpretable

Limitamos la profundidad y exigimos al menos 30 observaciones por hoja. No escalamos: los cortes conservarán las unidades y valores originales.

In [ ]:
tree_small = DecisionTreeClassifier(
    criterion="gini", max_depth=3, min_samples_leaf=30, random_state=RANDOM_STATE
)
tree_small.fit(X_train, y_train)

plt.figure(figsize=(20, 10))
plot_tree(
    tree_small, feature_names=X.columns, class_names=["Permanece", "Churn"],
    filled=True, rounded=True, proportion=True, precision=2, fontsize=9
)
plt.title("Árbol de churn: reglas en la escala original")
plt.show()

Para leer cada nodo: la primera línea es la regla; `gini` mide impureza; `samples` muestra la proporción de entrenamiento; `value` contiene la distribución por clase y `class` la predicción mayoritaria.

In [ ]:
clients = X_test.iloc[:2]
predictions = pd.DataFrame({
    "real": y_test.iloc[:2].to_numpy(),
    "predicted": tree_small.predict(clients),
    "p_churn": tree_small.predict_proba(clients)[:, 1],
}, index=clients.index)
display(clients)
display(predictions)
tree_small.decision_path(clients).toarray()

## 6. Complejidad sin tocar test

Comparamos configuraciones mediante validación cruzada dentro de entrenamiento. No buscamos exhaustivamente el mejor árbol; observamos el compromiso entre flexibilidad e incertidumbre.

In [ ]:
candidates = {
    "profundidad_1": DecisionTreeClassifier(max_depth=1, min_samples_leaf=30, random_state=RANDOM_STATE),
    "profundidad_3": tree_small,
    "profundidad_6": DecisionTreeClassifier(max_depth=6, min_samples_leaf=15, random_state=RANDOM_STATE),
    "sin_restricción": DecisionTreeClassifier(random_state=RANDOM_STATE),
}
rows = []
for name, model in candidates.items():
    scores = cross_validate(model, X_train, y_train, cv=cv, scoring=scoring, return_train_score=True)
    rows.append({
        "modelo": name, "recall_train": scores["train_recall"].mean(),
        "recall_cv": scores["test_recall"].mean(), "f1_cv": scores["test_f1"].mean(),
    })
comparison = pd.DataFrame(rows).set_index("modelo").round(3)
comparison

**Reto 2.** Identifique evidencia de underfitting o overfitting. Luego cambie `criterion` a `entropy` o aumente `min_samples_leaf`. Justifique la comparación antes de mirar test.

## 7. Evaluación final

El árbol de profundidad 3 fue útil para aprender a leer reglas. Para la evaluación final elegimos entre los candidatos usando recall y F1 de validación, sin observar test. En estos datos, la profundidad 6 ofrece un mejor equilibrio predictivo. Ahora, y solo ahora, usamos test.

In [ ]:
final_tree = candidates["profundidad_6"]
final_tree.fit(X_train, y_train)
y_pred = final_tree.predict(X_test)
print(classification_report(y_test, y_pred, target_names=["Permanece", "Churn"], zero_division=0))
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred, display_labels=["Permanece", "Churn"], cmap="Greens"
)
plt.title("Evaluación final del árbol")
plt.show()

## 8. Cierre

Redacte entre cinco y ocho líneas: decisión apoyada, comparación con la referencia, error que permanece, dos reglas relevantes y una advertencia. Las reglas son asociaciones predictivas; no demuestran que intervenir una variable cause menor churn.